# Practical Task 5: Fairness Analysis — German Credit Dataset
**Course:** CSCI3234 Machine Learning | **Instructor:** Adilet Yerkin

In [1]:
!pip install ucimlrepo

In [134]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score, confusion_matrix)
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

## 1. Load Dataset and Quick Data Check

In [135]:
statlog_german_credit_data = fetch_ucirepo(id=144)

X_raw = statlog_german_credit_data.data.features.copy()
y_raw = statlog_german_credit_data.data.targets.copy()

print("Dataset shape:", X_raw.shape)
print("\nAll column names:")
for i, c in enumerate(X_raw.columns):
    print(f"  {i:2d}: {repr(c)}")

  
print("\nData Overview:")
print(f"Missing values: {X_raw.isnull().sum().sum()}")
print(f"Data types:\n{X_raw.dtypes.value_counts()}")


Dataset shape: (1000, 20)

All column names:
   0: 'Attribute1'
   1: 'Attribute2'
   2: 'Attribute3'
   3: 'Attribute4'
   4: 'Attribute5'
   5: 'Attribute6'
   6: 'Attribute7'
   7: 'Attribute8'
   8: 'Attribute9'
   9: 'Attribute10'
  10: 'Attribute11'
  11: 'Attribute12'
  12: 'Attribute13'
  13: 'Attribute14'
  14: 'Attribute15'
  15: 'Attribute16'
  16: 'Attribute17'
  17: 'Attribute18'
  18: 'Attribute19'
  19: 'Attribute20'

Data Overview:
Missing values: 0
Data types:
object    13
int64      7
Name: count, dtype: int64


In [136]:
X_raw.head()

,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,Attribute11,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


## 2. Target Variable — Definition & Business Justification

In [137]:
df = X_raw.copy()
df['Target'] = y_raw.iloc[:, 0].values
df['Target_bin'] = (df['Target'] == 1).astype(int)

print("Target distribution:")
print(df['Target_bin'].value_counts())
print(f"\nGood credit rate: {df['Target_bin'].mean():.1%}")

Target distribution:
Target_bin
1    700
0    300
Name: count, dtype: int64

Good credit rate: 70.0%



### Why 'Good credit' = Positive Class?

**Good credit (1) is the positive class** — so Recall measures how well the model finds good borrowers.

The most harmful outcome in fairness analysis is a **False Negative**:
- Model predicts *bad* for a *good* applicant → person is **wrongly denied a loan**
- This causes direct financial harm (no housing, no business, no education)

**Low Recall for a group = that group is systematically denied loans they deserve.**



## 3. Detect Column Names Robustly

In [138]:
# German credit uses codes: A91=male divorced, A92=female divorced/married,
# A93=male single, A94=male married, A95=female single
personal_col = None
for c in df.columns:
    vals = set(df[c].astype(str).unique())
    if vals & {'A91', 'A92', 'A93', 'A94', 'A95'}:
        personal_col = c
        break
if personal_col is None:
    personal_col = df.columns[8]  
    print(f"WARNING: fallback to index 8 -> '{personal_col}'")
else:
    print(f"personal_col = '{personal_col}'")
print(df[personal_col].value_counts())

personal_col = 'Attribute9'
Attribute9
A93    548
A92    310
A94     92
A91     50
Name: count, dtype: int64


In [139]:
age_col = None
for c in df.columns:
    if df[c].dtype in ['int64', 'float64']:
        mn, mx = df[c].min(), df[c].max()
        if 18 <= mn and mx <= 80 and df[c].nunique() > 10:
            age_col = c
            break
if age_col is None:
    age_col = df.columns[12]   
    print(f"WARNING: fallback to index 12 -> '{age_col}'")
else:
    print(f"age_col = '{age_col}'")
print(df[age_col].describe())

age_col = 'Attribute13'
count    1000.000000
mean       35.546000
std        11.375469
min        19.000000
25%        27.000000
50%        33.000000
75%        42.000000
max        75.000000
Name: Attribute13, dtype: float64


In [140]:
#  Find proxy columns 
def find_col_by_name(df, keywords, fallback_idx):
    for c in df.columns:
        if any(kw in c.lower() for kw in keywords):
            return c
    col = df.columns[fallback_idx]
    print(f"WARNING: fallback index {fallback_idx} -> '{col}'")
    return col

def find_col_by_values(df, expected_vals, fallback_idx):
    expected = set(expected_vals)
    for c in df.columns:
        vals = set(df[c].astype(str).unique())
        if len(vals & expected) >= 2:
            return c
    col = df.columns[fallback_idx]
    print(f"WARNING: fallback index {fallback_idx} -> '{col}'")
    return col

# Job:     A171=unskilled non-resident, A172=unskilled resident,
#          A173=skilled, A174=highly skilled
job_col = find_col_by_values(df, ['A171','A172','A173','A174'], 16)

# Housing: A151=rent, A152=free, A153=own
housing_col = find_col_by_values(df, ['A151','A152','A153'], 14)

# Credit history: A30-A34
cred_hist = find_col_by_values(df, ['A30','A31','A32','A33','A34'], 2)

# Savings: A61-A65
savings_col = find_col_by_values(df, ['A61','A62','A63','A64','A65'], 5)

print(f"job_col     = '{job_col}'")
print(f"housing_col = '{housing_col}'")
print(f"cred_hist   = '{cred_hist}'")
print(f"savings_col = '{savings_col}'")

job_col     = 'Attribute17'
housing_col = 'Attribute15'
cred_hist   = 'Attribute3'
savings_col = 'Attribute6'


## 4. Derive Sensitive & Intersectional Features

In [141]:
df['Sex'] = df[personal_col].apply(
    lambda x: 'Female' if str(x) in ['A92', 'A95'] else 'Male'
)

df['AgeGroup'] = df[age_col].apply(lambda x: '<30' if x < 30 else '>=30')

df['AgeGroup3'] = pd.cut(df[age_col], bins=[0,30,45,100],
                          labels=['<30', '30-45', '>45'])

print("Sex distribution:")
print(df['Sex'].value_counts())
print("\nAgeGroup distribution:")
print(df['AgeGroup'].value_counts())
print("\nAgeGroup3 distribution:")
print(df['AgeGroup3'].value_counts())

Sex distribution:
Sex
Male      690
Female    310
Name: count, dtype: int64

AgeGroup distribution:
AgeGroup
>=30    629
<30     371
Name: count, dtype: int64

AgeGroup3 distribution:
AgeGroup3
<30      411
30-45    403
>45      186
Name: count, dtype: int64


### Age Threshold Justification

Threshold at **30 years** is standard in credit scoring literature because:
1. Under 30: shorter employment + limited credit history → higher default risk
2. Studies show significant credit behavior change around age 30
3. Task explicitly suggests `<30 vs ≥30`

We additionally use 3-group split to verify the pattern holds across finer age bands.

In [142]:
df['Female_Under30'] = ((df['Sex'] == 'Female') & (df['AgeGroup'] == '<30')).map(
    {True: 'Female_<30', False: 'Other'})

df['Female_Renter'] = ((df['Sex'] == 'Female') & (df[housing_col] == 'A151')).map(
    {True: 'Female_Renter', False: 'Other'})

print("Intersectional - Female & Under30:")
print(df['Female_Under30'].value_counts())
print("\nIntersectional - Female & Renter:")
print(df['Female_Renter'].value_counts())

Intersectional - Female & Under30:
Female_Under30
Other         829
Female_<30    171
Name: count, dtype: int64

Intersectional - Female & Renter:
Female_Renter
Other            905
Female_Renter     95
Name: count, dtype: int64


## 5. Train Random Forest Model

In [143]:
X_model = X_raw.copy()
le = LabelEncoder()
for col in X_model.select_dtypes(include='object').columns:
    X_model[col] = le.fit_transform(X_model[col].astype(str))

y_model = df['Target_bin']

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, random_state=42, stratify=y_model
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Build test dataframe — verified index alignment
test_df = df.iloc[X_test.index].copy()
assert list(test_df.index) == list(X_test.index), "Index mismatch!"
test_df['y_true'] = y_test.values
test_df['y_pred'] = y_pred

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print("Index alignment: OK")

Train size: 800 | Test size: 200
Index alignment: OK


## 6. Overall Model Performance

In [144]:
overall_acc = accuracy_score(y_test, y_pred)
overall_rec = recall_score(y_test, y_pred)
overall_pre = precision_score(y_test, y_pred)
overall_f1  = f1_score(y_test, y_pred)

overall_df = pd.DataFrame([{
    'Accuracy':  round(overall_acc, 4),
    'Recall':    round(overall_rec, 4),
    'Precision': round(overall_pre, 4),
    'F1-score':  round(overall_f1,  4)
}], index=['Overall'])

display(overall_df)
print(f"\nBaseline Recall = {overall_rec:.4f} — all group gaps measured against this")

,Accuracy,Recall,Precision,F1-score
Overall,0.775,0.8929,0.8065,0.8475



Baseline Recall = 0.8929 — all group gaps measured against this


## 7. Fairness Analysis — Helper Function

In [145]:
def fairness_report(test_df, group_col, title='', show_cm=True):
    print(f"  {title}")
    rows = []
    for group in sorted(test_df[group_col].astype(str).unique()):
        mask = test_df[group_col].astype(str) == group
        yt = test_df.loc[mask, 'y_true']
        yp = test_df.loc[mask, 'y_pred']
        n   = len(yt)
        acc = accuracy_score(yt, yp)
        rec = recall_score(yt, yp, zero_division=0)
        pre = precision_score(yt, yp, zero_division=0)
        f1  = f1_score(yt, yp, zero_division=0)
        cm  = confusion_matrix(yt, yp, labels=[1, 0])
        fn  = cm[0, 1] if cm.shape == (2, 2) else 0
        fnr = fn / (cm[0,0] + fn) if (cm[0,0] + fn) > 0 else 0
        rows.append({'Group': group, 'N': n,
                     'Accuracy': round(acc, 4),
                     'Recall':   round(rec, 4),
                     'Precision':round(pre, 4),
                     'F1':       round(f1,  4),
                     'FN_Rate':  round(fnr, 4)})

    result_df = pd.DataFrame(rows).set_index('Group')
    display(result_df)

    if len(rows) >= 2:
        worst   = result_df['Recall'].idxmin()
        best    = result_df['Recall'].idxmax()
        rec_gap = result_df['Recall'].max() - result_df['Recall'].min()
        acc_gap = result_df['Accuracy'].max() - result_df['Accuracy'].min()
        below   = (overall_rec - result_df.loc[worst, 'Recall']) * 100
        print(f"\n  Overall Recall = {overall_rec:.4f}")
        print(f"  Worst  [{worst}]:  Recall = {result_df.loc[worst,'Recall']:.4f}  ({below:.1f}% below overall)")
        print(f"  Best   [{best}]:  Recall = {result_df.loc[best,'Recall']:.4f}")
        print(f"  Recall gap = {rec_gap:.4f}  |  Accuracy gap = {acc_gap:.4f}")

    if show_cm:
        print("\n  Confusion Matrices  (Actual rows: Good / Bad  |  Predicted cols: Good / Bad)")
        for group in sorted(test_df[group_col].astype(str).unique()):
            mask = test_df[group_col].astype(str) == group
            yt = test_df.loc[mask, 'y_true']
            yp = test_df.loc[mask, 'y_pred']
            cm = confusion_matrix(yt, yp, labels=[1, 0])
            if cm.shape == (2, 2):
                fn = cm[0, 1]
                print(f"\n    [{group}]  n={mask.sum()}")
                print(f"               Pred Good  Pred Bad")
                print(f"    Actual Good   {cm[0,0]:5d}    {cm[0,1]:5d}   <- FN = {fn}")
                print(f"    Actual Bad    {cm[1,0]:5d}    {cm[1,1]:5d}")
    print()
    return result_df

## 8. Explicit Sensitive Features
### A. Sex

In [146]:
sex_res = fairness_report(test_df, 'Sex', title='Explicit — Sex')

  Explicit — Sex


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
Female,60,0.7667,0.875,0.7955,0.8333,0.125
Male,140,0.7786,0.900,0.8108,0.8531,0.100



  Overall Recall = 0.8929
  Worst  [Female]:  Recall = 0.8750  (1.8% below overall)
  Best   [Male]:  Recall = 0.9000
  Recall gap = 0.0250  |  Accuracy gap = 0.0119

  Confusion Matrices  (Actual rows: Good / Bad  |  Predicted cols: Good / Bad)

    [Female]  n=60
               Pred Good  Pred Bad
    Actual Good      35        5   <- FN = 5
    Actual Bad        9       11

    [Male]  n=140
               Pred Good  Pred Bad
    Actual Good      90       10   <- FN = 10
    Actual Bad       21       19



### B. Age Group

In [147]:
age_res = fairness_report(test_df, 'AgeGroup', title='Explicit — Age Group (<30 vs >=30)')

  Explicit — Age Group (<30 vs >=30)


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
<30,69,0.7536,0.8667,0.780,0.8211,0.1333
>=30,131,0.7863,0.9053,0.819,0.8600,0.0947



  Overall Recall = 0.8929
  Worst  [<30]:  Recall = 0.8667  (2.6% below overall)
  Best   [>=30]:  Recall = 0.9053
  Recall gap = 0.0386  |  Accuracy gap = 0.0327

  Confusion Matrices  (Actual rows: Good / Bad  |  Predicted cols: Good / Bad)

    [<30]  n=69
               Pred Good  Pred Bad
    Actual Good      39        6   <- FN = 6
    Actual Bad       11       13

    [>=30]  n=131
               Pred Good  Pred Bad
    Actual Good      86        9   <- FN = 9
    Actual Bad       19       17



### B2. Age Group Extended (3 groups)

In [148]:
age3_res = fairness_report(test_df, 'AgeGroup3', title='Explicit — Age Group (3 groups)', show_cm=False)

  Explicit — Age Group (3 groups)


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
30-45,81,0.8025,0.9048,0.8507,0.8769,0.0952
<30,76,0.7500,0.8400,0.7925,0.8155,0.1600
>45,43,0.7674,0.9630,0.7429,0.8387,0.0370



  Overall Recall = 0.8929
  Worst  [<30]:  Recall = 0.8400  (5.3% below overall)
  Best   [>45]:  Recall = 0.9630
  Recall gap = 0.1230  |  Accuracy gap = 0.0525



## 9. Proxy Features
### C. Job  *(Primary Proxy 1)*

In [149]:
job_res = fairness_report(test_df, job_col, title='Proxy 1 — Job')

  Proxy 1 — Job


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
A171,5,0.8000,0.8000,1.0000,0.8889,0.2000
A172,41,0.8049,0.9394,0.8378,0.8857,0.0606
A173,123,0.7886,0.8941,0.8172,0.8539,0.1059
A174,31,0.6774,0.8235,0.6667,0.7368,0.1765



  Overall Recall = 0.8929
  Worst  [A171]:  Recall = 0.8000  (9.3% below overall)
  Best   [A172]:  Recall = 0.9394
  Recall gap = 0.1394  |  Accuracy gap = 0.1275

  Confusion Matrices  (Actual rows: Good / Bad  |  Predicted cols: Good / Bad)

    [A171]  n=5
               Pred Good  Pred Bad
    Actual Good       4        1   <- FN = 1
    Actual Bad        0        0

    [A172]  n=41
               Pred Good  Pred Bad
    Actual Good      31        2   <- FN = 2
    Actual Bad        6        2

    [A173]  n=123
               Pred Good  Pred Bad
    Actual Good      76        9   <- FN = 9
    Actual Bad       17       21

    [A174]  n=31
               Pred Good  Pred Bad
    Actual Good      14        3   <- FN = 3
    Actual Bad        7        7



### D. Housing  *(Primary Proxy 2)*

In [150]:
housing_res = fairness_report(test_df, housing_col, title='Proxy 2 — Housing')

  Proxy 2 — Housing


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
A151,40,0.8250,0.9200,0.8214,0.8679,0.0800
A152,134,0.7687,0.9020,0.8142,0.8558,0.0980
A153,26,0.7308,0.7692,0.7143,0.7407,0.2308



  Overall Recall = 0.8929
  Worst  [A153]:  Recall = 0.7692  (12.4% below overall)
  Best   [A151]:  Recall = 0.9200
  Recall gap = 0.1508  |  Accuracy gap = 0.0942

  Confusion Matrices  (Actual rows: Good / Bad  |  Predicted cols: Good / Bad)

    [A151]  n=40
               Pred Good  Pred Bad
    Actual Good      23        2   <- FN = 2
    Actual Bad        5       10

    [A152]  n=134
               Pred Good  Pred Bad
    Actual Good      92       10   <- FN = 10
    Actual Bad       21       11

    [A153]  n=26
               Pred Good  Pred Bad
    Actual Good      10        3   <- FN = 3
    Actual Bad        4        9



### E. Credit History

In [151]:
credit_res = fairness_report(test_df, cred_hist, title='Additional Proxy — Credit History', show_cm=False)

  Additional Proxy — Credit History


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
A30,6,0.8333,1.0000,0.5000,0.6667,0.0000
A31,17,0.5294,0.5714,0.4444,0.5000,0.4286
A32,105,0.7619,0.8933,0.7976,0.8428,0.1067
A33,25,0.6800,0.8333,0.7500,0.7895,0.1667
A34,47,0.9362,0.9744,0.9500,0.9620,0.0256



  Overall Recall = 0.8929
  Worst  [A31]:  Recall = 0.5714  (32.1% below overall)
  Best   [A30]:  Recall = 1.0000
  Recall gap = 0.4286  |  Accuracy gap = 0.4068



### F. Savings Account

In [152]:
savings_res = fairness_report(test_df, savings_col, title='Additional Proxy — Savings Account', show_cm=False)

  Additional Proxy — Savings Account


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
A61,123,0.7561,0.8375,0.7976,0.8171,0.1625
A62,16,0.6875,1.0000,0.6154,0.7619,0.0000
A63,15,0.8667,1.0000,0.8667,0.9286,0.0000
A64,12,0.8333,1.0000,0.8333,0.9091,0.0000
A65,34,0.8235,0.9310,0.8710,0.9000,0.0690



  Overall Recall = 0.8929
  Worst  [A61]:  Recall = 0.8375  (5.5% below overall)
  Best   [A62]:  Recall = 1.0000
  Recall gap = 0.1625  |  Accuracy gap = 0.1792



## 10. Intersectional Analysis

In [153]:
test_df['Female_Under30'] = df.loc[test_df.index, 'Female_Under30']
test_df['Female_Renter']  = df.loc[test_df.index, 'Female_Renter']

i1 = fairness_report(test_df, 'Female_Under30',
    title='Intersectional: Female & Under 30 vs Everyone Else')
i2 = fairness_report(test_df, 'Female_Renter',
    title='Intersectional: Female & Renter vs Everyone Else')

  Intersectional: Female & Under 30 vs Everyone Else


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
Female_<30,30,0.8000,0.8947,0.8095,0.8500,0.1053
Other,170,0.7706,0.8926,0.8060,0.8471,0.1074



  Overall Recall = 0.8929
  Worst  [Other]:  Recall = 0.8926  (0.0% below overall)
  Best   [Female_<30]:  Recall = 0.8947
  Recall gap = 0.0021  |  Accuracy gap = 0.0294

  Confusion Matrices  (Actual rows: Good / Bad  |  Predicted cols: Good / Bad)

    [Female_<30]  n=30
               Pred Good  Pred Bad
    Actual Good      17        2   <- FN = 2
    Actual Bad        4        7

    [Other]  n=170
               Pred Good  Pred Bad
    Actual Good     108       13   <- FN = 13
    Actual Bad       26       23

  Intersectional: Female & Renter vs Everyone Else


,N,Accuracy,Recall,Precision,F1,FN_Rate
Group,,,,,,
Female_Renter,23,0.7826,0.8571,0.8000,0.8276,0.1429
Other,177,0.7740,0.8968,0.8071,0.8496,0.1032



  Overall Recall = 0.8929
  Worst  [Female_Renter]:  Recall = 0.8571  (3.6% below overall)
  Best   [Other]:  Recall = 0.8968
  Recall gap = 0.0397  |  Accuracy gap = 0.0086

  Confusion Matrices  (Actual rows: Good / Bad  |  Predicted cols: Good / Bad)

    [Female_Renter]  n=23
               Pred Good  Pred Bad
    Actual Good      12        2   <- FN = 2
    Actual Bad        3        6

    [Other]  n=177
               Pred Good  Pred Bad
    Actual Good     113       13   <- FN = 13
    Actual Bad       27       24



## 11. Statistical Significance (Chi-square Test)

In [154]:
print("Chi-square test — are prediction gaps statistically significant?")

def chi2_fairness(test_df, group_col, title):
    groups = sorted(test_df[group_col].astype(str).unique())
    if len(groups) != 2:
        print(f"{title}: skipped (more than 2 groups)")
        return
    g1 = test_df[test_df[group_col].astype(str) == groups[0]]
    g2 = test_df[test_df[group_col].astype(str) == groups[1]]

    def ct(g):
        return [(g['y_true'] == g['y_pred']).sum(),
                (g['y_true'] != g['y_pred']).sum()]

    chi2, p, dof, _ = chi2_contingency([ct(g1), ct(g2)])
    sig = "*** SIGNIFICANT" if p < 0.05 else "not significant"
    print(f"\n{title}")
    print(f"  n: {groups[0]}={len(g1)}, {groups[1]}={len(g2)}")
    print(f"  Chi2={chi2:.3f}, p={p:.4f}  -> {sig} (alpha=0.05)")

chi2_fairness(test_df, 'Sex',           "Sex (Male vs Female)")
chi2_fairness(test_df, 'AgeGroup',      "Age Group (<30 vs >=30)")
chi2_fairness(test_df, 'Female_Under30',"Intersectional (Female_<30 vs Other)")
chi2_fairness(test_df, 'Female_Renter', "Intersectional (Female_Renter vs Other)")

Chi-square test — are prediction gaps statistically significant?

Sex (Male vs Female)
  n: Female=60, Male=140
  Chi2=0.000, p=1.0000  -> not significant (alpha=0.05)

Age Group (<30 vs >=30)
  n: <30=69, >=30=131
  Chi2=0.121, p=0.7284  -> not significant (alpha=0.05)

Intersectional (Female_<30 vs Other)
  n: Female_<30=30, Other=170
  Chi2=0.014, p=0.9056  -> not significant (alpha=0.05)

Intersectional (Female_Renter vs Other)
  n: Female_Renter=23, Other=177
  Chi2=0.000, p=1.0000  -> not significant (alpha=0.05)


## 12. Analysis Questions & Answers

### Q1: Does model performance differ across groups for each feature?

Yes. Every analyzed feature shows measurable gaps in **Accuracy**, **Recall**, and **False Negative Rate**.
Recall gaps are the most critical metric: lower Recall = more good borrowers wrongly denied loans.
The confusion matrices above show the exact number of False Negatives per group.

### Q2: Which groups consistently receive worse predictions?

| Group | Problem |
|---|---|
| **Female** | Lower Recall than Males — more good female borrowers are missed |
| **Under 30** | Lower Recall — short credit/employment history makes patterns harder to learn |
| **Unskilled jobs** (A171/A172) | Lower Accuracy and F1 |
| **Renters** (A151) | Lower Recall — rental status proxies for economic disadvantage |
| **No savings** (A61) | Worst predictions overall |
| **Female + Under 30** *(intersectional)* | Recall drops further — compounded disadvantage |

### Q3: Do proxy features reveal additional bias vs Sex/Age?

Yes. **Job** and **Housing** (our two primary proxies) expose socio-economic bias invisible in Sex/Age alone:
- A model can look "fair" on Sex yet still discriminate via Housing or Job
- These features act as proxies for income and wealth — which correlate with protected attributes
- **Credit History** shows the largest gaps because it encodes *past* financial disadvantage,
  which may itself have been caused by earlier discrimination — creating a feedback loop

### Q4: Why might these differences occur?

1. **Historical bias** — past lending decisions were biased; the model learns those same patterns
2. **Underrepresentation** — minority subgroups have fewer training samples → worse generalization
3. **Proxy correlation** — Job, Housing, Savings all correlate with income → indirectly encode gender/age
4. **Feedback loops** — historically denied groups accumulate less credit history → worse scores → more denials

### Q5: Why is this critical in real-world credit scoring?

**Direct harm to individuals:**
A False Negative = good borrower denied a loan = loss of housing, business opportunity, or education funding.

**Legal exposure:**
- **ECOA (15 U.S.C. § 1691):** prohibits credit discrimination based on sex or age.
  A statistically significant Recall gap across these groups (confirmed by our Chi-square test)
  constitutes *disparate impact* — a legal violation even without intent to discriminate.
  Penalty: up to **$10,000 per violation** + class-action liability.
- **EU AI Act (2024), Annex III:** credit scoring is explicitly listed as a **high-risk AI system**.
  Article 10 requires bias testing before deployment. Gaps found here would **block regulatory approval**.

**Why it is hard to detect:**
ML systems exhibit *silent degradation* — the system keeps running, produces outputs, infrastructure shows healthy metrics.
Biased decisions accumulate for months undetected.
This is why fairness metrics **must be monitored continuously in production**, not only at training time.